# ML Mathematics Playground: 02. Calculus & Optimization

Optimization is the engine that allows AI models to learn. Gradient Descent uses calculus to iteratively adjust parameters (weights) to minimize error (loss).

In this notebook, we will explore:
1. **The Tangent Limit**: How derivatives measure slope.
2. **1D Gradient Descent**: Minimizing cost functions dynamically.
3. **3D Optimization Landscapes**: Visualizing partial derivatives and saddle points.


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, interactive

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (8, 5)


## Section 1: The Tangent Limit

The derivative of a function $f(x)$ at $x$ measures the instantaneous rate of change. It is defined as the limit of the secant line slope as $\Delta x 	o 0$:
$$f'(x) = \lim_{\Delta x \to 0} \frac{f(x + \Delta x) - f(x)}{\Delta x}$$

Drag the $\Delta x$ slider below to see the secant line approach the true tangent line for $f(x) = x^2$ at $x_0 = 2$.


In [ ]:
def f(x):
    return x**2

def plot_tangent_limit(dx):
    x0 = 2.0
    y0 = f(x0)
    
    x_vals = np.linspace(-1, 5, 200)
    y_vals = f(x_vals)
    
    fig, ax = plt.subplots()
    ax.plot(x_vals, y_vals, label='f(x) = x^2', color='blue', lw=2)
    ax.scatter([x0], [y0], color='red', s=100, zorder=5, label='Point (x0, y0)')
    
    # Target point
    x1 = x0 + dx
    y1 = f(x1)
    ax.scatter([x1], [y1], color='orange', s=80, zorder=5, label='x0 + dx')
    
    # Secant line
    slope = (y1 - y0) / dx if dx != 0 else 4.0
    secant_y = slope * (x_vals - x0) + y0
    ax.plot(x_vals, secant_y, '--', color='orange', label=f'Secant slope = {slope:.3f}')
    
    # True tangent
    true_slope = 4.0
    tangent_y = true_slope * (x_vals - x0) + y0
    ax.plot(x_vals, tangent_y, color='green', alpha=0.6, label='True Tangent slope = 4.000')
    
    ax.set_ylim(-2, 25)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend()
    ax.set_title(f'Approaching the Derivative (dx = {dx:.3f})')
    plt.show()

interact(plot_tangent_limit, dx=widgets.FloatSlider(value=2.0, min=0.01, max=3.0, step=0.05, description='dx'));


## Section 2: 1D Gradient Descent Simulator

To minimize a loss function, gradient descent updates $x$ using the rule:
$$x_{new} = x_{old} - \alpha \cdot f'(x_{old})$$
where $\alpha$ is the **learning rate**.

Let's test this on two functions:
1. **Convex**: $f(x) = x^2$ (one clear minimum).
2. **Non-Convex**: $f(x) = x^4 - 2x^2$ (multiple local minima).

Change the learning rate $\alpha$ and see what happens when it is too small (slow learning) or too large (exploding steps / oscillation).


In [ ]:
def g_convex(x):
    return x**2
def dg_convex(x):
    return 2*x

def g_nonconvex(x):
    return x**4 - 2*x**2
def dg_nonconvex(x):
    return 4*x**3 - 4*x

def simulate_gd(func_type, start_x, lr, steps):
    if func_type == 'Convex (x^2)':
        f, df = g_convex, dg_convex
        x_range = np.linspace(-3, 3, 200)
    else:
        f, df = g_nonconvex, dg_nonconvex
        x_range = np.linspace(-1.8, 1.8, 200)
        
    # Run Gradient Descent
    history_x = [start_x]
    history_y = [f(start_x)]
    curr_x = start_x
    for _ in range(steps):
        grad = df(curr_x)
        curr_x = curr_x - lr * grad
        history_x.append(curr_x)
        history_y.append(f(curr_x))
        
    fig, ax = plt.subplots()
    ax.plot(x_range, f(x_range), color='blue', lw=2, label='Cost Function')
    ax.plot(history_x, history_y, 'ro-', label='GD Steps', markersize=6, alpha=0.8)
    ax.scatter([curr_x], [f(curr_x)], color='yellow', s=150, zorder=5, edgecolor='black', label='Final Point')
    
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend()
    ax.set_title(f'Gradient Descent (Final x = {curr_x:.4f})')
    plt.show()

interact(simulate_gd,
         func_type=widgets.Dropdown(options=['Convex (x^2)', 'Non-Convex (x^4 - 2x^2)'], value='Convex (x^2)', description='Function'),
         start_x=widgets.FloatSlider(value=2.5, min=-3.0, max=3.0, step=0.1, description='Start x0'),
         lr=widgets.FloatSlider(value=0.1, min=0.01, max=1.1, step=0.02, description='Learning Rate'),
         steps=widgets.IntSlider(value=10, min=1, max=50, step=1, description='Steps'));


## Section 3: 3D Optimization Landscapes

In ML, models have thousands of parameters. Optimization happens in high-dimensional space.
Here we visualize a 3D loss surface $f(x, y) = x^2 + 2y^2$. The gradient vector is:
$$\nabla f(x, y) = \begin{pmatrix} \frac{\partial f}{\partial x} \\ \frac{\partial f}{\partial y} \end{pmatrix} = \begin{pmatrix} 2x \\ 4y \end{pmatrix}$$
The gradient always points in the direction of **steepest ascent** (upwards), so we move opposite to it.


In [ ]:
from mpl_toolkits.mplot3d import Axes3D

def cost_3d(x, y):
    return x**2 + 2*y**2

def plot_gd_3d(start_x, start_y, lr, steps):
    # Run 3D GD
    hx, hy = [start_x], [start_y]
    hz = [cost_3d(start_x, start_y)]
    
    cx, cy = start_x, start_y
    for _ in range(steps):
        gx = 2 * cx
        gy = 4 * cy
        cx = cx - lr * gx
        cy = cy - lr * gy
        hx.append(cx)
        hy.append(cy)
        hz.append(cost_3d(cx, cy))
        
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')
    
    # Create Surface
    X = np.linspace(-3, 3, 50)
    Y = np.linspace(-3, 3, 50)
    X, Y = np.meshgrid(X, Y)
    Z = cost_3d(X, Y)
    
    ax.plot_surface(X, Y, Z, cmap='viridis', alpha=0.5, edgecolor='none')
    ax.plot(hx, hy, hz, 'ro-', lw=2, markersize=5, label='GD Path', zorder=10)
    
    ax.set_xlabel('Parameter X')
    ax.set_ylabel('Parameter Y')
    ax.set_zlabel('Loss / Cost')
    ax.set_title('3D Gradient Descent Landscape')
    ax.legend()
    plt.show()

interact(plot_gd_3d,
         start_x=widgets.FloatSlider(value=2.5, min=-3.0, max=3.0, step=0.1, description='Start X'),
         start_y=widgets.FloatSlider(value=2.5, min=-3.0, max=3.0, step=0.1, description='Start Y'),
         lr=widgets.FloatSlider(value=0.1, min=0.01, max=0.6, step=0.02, description='Learning Rate'),
         steps=widgets.IntSlider(value=10, min=1, max=40, step=1, description='Steps'));
